# ElonMealsDB production-history analysis

## tl;dr

- **Build a dedicated Menu Rotation section first.** Across 38 usable live menu days, the median menu changed by **56.6%** versus the previous observed menu. The history contains **19 every-day staples** and **177 one-day items**, which supports a clear reliable-staples-versus-discoveries experience.
- **Treat current scale as a summer baseline only.** The live period contains only 1 restaurant on any day (Lakeside Dining Hall), so it cannot estimate school-term concurrency or cross-restaurant choice.
- **Add a visible freshness state before expanding the analytics.** Only 38 of 44 calendar dates had usable menus (86.4%); the latest successful service date at extraction was 2026-08-19.
- **Do not present missing allergen flags as verified safety.** The current schema cannot distinguish “not listed” from “confirmed absent,” and zero-valued nutrition is also used for some implausible items. Fix those semantics before making historical dietary or allergen claims.

## Context & Methods

### Decision

Identify evidence-backed tables, visualizations, and sections worth adding to the production website, while separating features supported by the current summer history from features that require school-term data.

### Key Assumptions

- Source of truth: the production MySQL database on `macmaxi`, read through the container's read-only API account.
- Analysis window: July 8 through August 20, 2026. July 1 is excluded because it is the bundled deterministic seed date, not a comparable live observation.
- A usable menu day has at least one stored food appearance. Meal hours without stations or food remain visible in data-quality checks.
- Menu appearances and meal windows are date-specific. Food nutrition and flags are global upserts, so they are latest-known attributes—not historical snapshots.
- The entire window occurred during summer break. Day-of-week and concurrency results are descriptive of summer operations only.
- “No flagged top-9 allergens” means none of the nine Boolean fields is true; the schema does not preserve an explicit unknown state.

In [1]:
from analysis.elonmealsdb_history_analysis import load_and_analyze
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)
bundle = load_and_analyze()
summary = bundle.summary
summary

{'as_of_date': '2026-08-20',
 'live_start': '2026-07-08',
 'calendar_days_in_scope': 44,
 'days_with_restaurant_rows': 40,
 'days_with_usable_menus': 38,
 'calendar_coverage_rate': 0.9090909090909091,
 'usable_menu_coverage_rate': 0.8636363636363636,
 'live_restaurant_names': ['Lakeside Dining Hall'],
 'max_restaurants_in_a_day': 1,
 'distinct_appeared_foods': 458,
 'total_food_appearances': 3377,
 'median_daily_distinct_foods': 66.0,
 'daily_foods_q1': 47.0,
 'daily_foods_q3': 77.75,
 'min_daily_distinct_foods': 25,
 'max_daily_distinct_foods': 103,
 'median_item_service_days': 2.0,
 'single_day_items': 177,
 'single_day_item_share': 0.3864628820960699,
 'items_seen_on_10_plus_days': 57,
 'items_seen_every_valid_day': 19,
 'median_jaccard_vs_previous': 0.4338235294117647,
 'median_menu_turnover_vs_previous': 0.5661764705882353,
 'median_items_added_vs_previous': 25.0,
 'median_items_removed_vs_previous': 27.0,
 'median_vegetarian_share': 0.8492003998000999,
 'median_vegan_share': 0.62

## Data

The production history contains a final stored menu for each successful service date, plus every scraper attempt. Repeated imports replace a date's restaurant hierarchy; they do not preserve multiple versions of that day's menu.

In [2]:
daily_view = bundle.daily[[
    'service_date', 'weekday', 'restaurants', 'meals', 'stations',
    'appearances', 'distinct_foods', 'first_open', 'last_close'
]].copy()
daily_view.tail(12)

,service_date,weekday,restaurants,meals,stations,appearances,distinct_foods,first_open,last_close
29,2026-08-06,Thursday,1,3,15,138,85,07:00,19:00
30,2026-08-07,Friday,1,3,14,150,95,07:00,19:00
31,2026-08-08,Saturday,1,3,5,45,38,07:00,19:00
32,2026-08-09,Sunday,1,3,5,48,42,07:00,19:00
33,2026-08-10,Monday,1,3,15,148,100,07:00,19:00
34,2026-08-11,Tuesday,1,3,14,148,88,07:00,19:00
35,2026-08-12,Wednesday,1,3,14,156,103,09:00,19:00
36,2026-08-13,Thursday,1,3,14,156,103,09:00,19:30
37,2026-08-14,Friday,1,3,14,142,92,07:00,18:45
38,2026-08-17,Monday,1,2,10,111,62,11:00,19:30


## Results

### The history supports rotation features, not school-term concurrency

The current database has enough repeated dates to measure item and station recurrence. It does not yet have the multi-restaurant breadth needed to validate an open-venue concurrency visualization, because every live stored date contains Lakeside Dining Hall only.

In [3]:
coverage_table = pd.DataFrame([
    {'Measure': 'Calendar dates in scope', 'Value': summary['calendar_days_in_scope']},
    {'Measure': 'Dates with restaurant rows', 'Value': summary['days_with_restaurant_rows']},
    {'Measure': 'Dates with usable food menus', 'Value': summary['days_with_usable_menus']},
    {'Measure': 'Maximum restaurants in one live date', 'Value': summary['max_restaurants_in_a_day']},
    {'Measure': 'Distinct foods seen in final live snapshots', 'Value': summary['distinct_appeared_foods']},
    {'Measure': 'Stored food appearances', 'Value': summary['total_food_appearances']},
])
coverage_table

,Measure,Value
0,Calendar dates in scope,44
1,Dates with restaurant rows,40
2,Dates with usable food menus,38
3,Maximum restaurants in one live date,1
4,Distinct foods seen in final live snapshots,458
5,Stored food appearances,3377


### Stable staples and high-turnover menus can coexist

A median daily menu contained 66 distinct foods, but overlap with the previous observed menu was only 43.4%. The useful product story is therefore not simply “the menu changes”; it is that a small stable core sits underneath a much larger rotating layer.

In [4]:
recurrence_bins = pd.cut(
    bundle.recurrence['service_days'],
    [0, 1, 2, 4, 9, 19, 38],
    labels=['1 day', '2 days', '3–4 days', '5–9 days', '10–19 days', '20–38 days'],
).value_counts(sort=False).rename_axis('Observed service days').reset_index(name='Foods')
recurrence_bins

,Observed service days,Foods
0,1 day,177
1,2 days,80
2,3–4 days,81
3,5–9 days,63
4,10–19 days,18
5,20–38 days,39


In [5]:
bundle.recurrence.head(25)[[
    'short_name', 'service_days', 'first_seen', 'last_seen', 'share_of_valid_days'
]]

,short_name,service_days,first_seen,last_seen,share_of_valid_days
42,Balsamic Vinaigrette,38,2026-07-08,2026-08-19,1.000000
44,Black Raisins,38,2026-07-08,2026-08-19,1.000000
37,Broccoli Florets,38,2026-07-08,2026-08-19,1.000000
40,Caesar Dressing,38,2026-07-08,2026-08-19,1.000000
31,Chickpeas,38,2026-07-08,2026-08-19,1.000000
35,Chilled Quinoa,38,2026-07-08,2026-08-19,1.000000
33,Chopped Hard-Cooked Egg,38,2026-07-08,2026-08-19,1.000000
21,Dried Cranberries,38,2026-07-08,2026-08-19,1.000000
30,Grape Tomatoes,38,2026-07-08,2026-08-19,1.000000
6,Grilled Chicken,38,2026-07-08,2026-08-19,1.000000


In [6]:
bundle.similarity.iloc[1:].sort_values('menu_turnover_vs_previous', ascending=False).head(12)[[
    'service_date', 'previous_valid_date', 'distinct_foods',
    'menu_turnover_vs_previous', 'added_vs_previous', 'removed_vs_previous', 'first_seen_in_history'
]]

,service_date,previous_valid_date,distinct_foods,menu_turnover_vs_previous,added_vs_previous,removed_vs_previous,first_seen_in_history
24,2026-08-04,2026-08-03,87,0.812500,66,25,36
28,2026-08-08,2026-08-07,38,0.801802,16,73,5
30,2026-08-10,2026-08-09,100,0.786325,75,17,18
37,2026-08-19,2026-08-18,60,0.778947,39,35,10
16,2026-07-27,2026-07-25,38,0.755814,17,48,7
36,2026-08-18,2026-08-17,56,0.717391,30,36,9
5,2026-07-13,2026-07-12,73,0.693333,50,2,20
19,2026-07-30,2026-07-29,54,0.683544,29,25,14
3,2026-07-11,2026-07-10,25,0.676923,4,40,4
35,2026-08-17,2026-08-14,62,0.672414,24,54,9


### Station behavior is legible enough for a rotation table

Among station-and-meal series observed on at least 10 days, seven changed on at least 80% of comparable appearances, while three changed on no more than 30%. That split can be translated into reader-friendly labels such as `Usually stable` and `Rotates often`.

In [7]:
bundle.station_rotation[bundle.station_rotation['service_days'] >= 10][[
    'meal_period', 'station', 'service_days', 'average_items',
    'distinct_menu_signatures', 'comparable_days', 'changed_days', 'change_rate'
]].sort_values(['change_rate', 'service_days'], ascending=[False, False])

,meal_period,station,service_days,average_items,distinct_menu_signatures,comparable_days,changed_days,change_rate
12,Lunch,allgood,34,6.264706,25,33,32,0.969697
18,Lunch,the world,31,10.419355,27,30,29,0.966667
6,Dinner,allgood,25,5.240000,19,24,23,0.958333
7,Dinner,comfort kitchen,20,6.250000,18,19,18,0.947368
13,Lunch,comfort kitchen,28,7.928571,24,27,25,0.925926
11,Dinner,the world,12,6.166667,11,11,10,0.909091
1,Breakfast,comfort kitchen,11,8.090909,8,10,9,0.900000
16,Lunch,street,12,12.000000,5,11,8,0.727273
4,Breakfast,griddle,22,9.454545,8,21,14,0.666667
15,Lunch,greens & grains,36,19.500000,6,35,9,0.257143


### Dietary breadth is promising, but safety semantics need repair

The latest-known food attributes produce large vegetarian, vegan, and gluten-free shares. Those are useful as exploratory menu labels. The allergen fields are different: all false currently means either no allergen was listed or no allergen information was captured, so the data cannot support a verified `safe` claim.

In [8]:
dietary_summary = pd.DataFrame([
    {'Label': 'Vegetarian', 'Median share of daily distinct foods': summary['median_vegetarian_share']},
    {'Label': 'Vegan', 'Median share of daily distinct foods': summary['median_vegan_share']},
    {'Label': 'Gluten-free flag', 'Median share of daily distinct foods': summary['median_gluten_free_share']},
    {'Label': 'No top-9 flag present (not verified allergen-free)', 'Median share of daily distinct foods': summary['median_no_top9_share']},
])
dietary_summary

,Label,Median share of daily distinct foods
0,Vegetarian,0.849200
1,Vegan,0.627016
2,Gluten-free flag,0.872076
3,No top-9 flag present (not verified allergen-f...,0.681818


### Reliability is good enough for discovery, not yet invisible enough for trust

The scheduler recorded 188 successful and 77 failed attempts through August 20. Repeated retries recovered many dates, but four dates in the window had no successful run, and two stored dates had meal hours but no food menu. In addition, 16 live food IDs carry zero calories and those rows appear on 32 of the 38 usable dates.

In [9]:
quality_table = pd.DataFrame([
    {'Check': 'Duplicate meal grains', 'Result': summary['quality']['duplicate_meal_grains'], 'Interpretation': 'Pass'},
    {'Check': 'Duplicate station grains', 'Result': summary['quality']['duplicate_station_grains'], 'Interpretation': 'Pass'},
    {'Check': 'Invalid meal windows', 'Result': summary['quality']['invalid_meal_windows'], 'Interpretation': 'Pass'},
    {'Check': 'Meals without stations', 'Result': summary['quality']['meals_without_stations'], 'Interpretation': 'Caveat'},
    {'Check': 'Live food IDs with zero calories', 'Result': summary['quality']['live_appeared_foods_with_zero_calories'], 'Interpretation': 'Needs semantic fix'},
    {'Check': 'Live dates affected by zero-calorie rows', 'Result': summary['quality']['live_dates_with_zero_calorie_items'], 'Interpretation': 'Needs semantic fix'},
])
quality_table

,Check,Result,Interpretation
0,Duplicate meal grains,0,Pass
1,Duplicate station grains,0,Pass
2,Invalid meal windows,0,Pass
3,Meals without stations,13,Caveat
4,Live food IDs with zero calories,16,Needs semantic fix
5,Live dates affected by zero-calorie rows,32,Needs semantic fix


In [10]:
failed_dates = bundle.run_daily[~bundle.run_daily['had_success']][[
    'target_date', 'attempts', 'failed_attempts', 'last_attempt'
]]
failed_dates

,target_date,attempts,failed_attempts,last_attempt
18,2026-07-26,6,6,2026-07-26 19:15:00
38,2026-08-15,6,6,2026-08-15 19:15:00
39,2026-08-16,6,6,2026-08-16 19:15:00
43,2026-08-20,4,4,2026-08-20 10:30:00


## Takeaways

### 1. Ship now: a separate Menu Rotation section

Recommended contents:

- A `What's different today?` summary with added, removed, first-seen, and returning-after-a-gap counts.
- A compact `Reliable staples` table: food, meal/station, days available in the last 14 or 28 usable service days, and last seen.
- A `Rotating stations` table: restaurant, meal, station, `Usually stable`/`Rotates often`, current item count, and last menu change.
- A favorite-level badge such as `Seen 12 of the last 14 menu days`; use observed menu days as the denominator, not calendar days.

Best primary visual: a 14- or 28-column presence heatmap for a small, user-selected set of foods or favorites. For campus-wide history, use ranked bars or a table instead of hundreds of heatmap rows.

### 2. Prepare now, activate during term: Where can I eat now?

When multi-restaurant data is successfully stored, add:

- A stepped `restaurants open by time` chart for the selected date.
- An `Open now` comparison table with restaurant, closes at, meals/stations, total items, vegetarian/vegan/gluten-free counts, and freshness.
- Filters that update the count to `restaurants with at least one matching option`, not just restaurants physically open.

Do not use the current one-restaurant summer history to set benchmarks. Wait until at least four complete in-session weeks before adding weekday norms or claims about peak choice.

### 3. Add a small freshness contract before scaling the public analytics

Display `Menu updated`, `Latest attempt`, and a plain-language state such as `Current`, `Using last successful menu`, or `Menu unavailable`. A partial failure should not make otherwise valid restaurants disappear. The importer currently rejects the entire date if any advertised meal has no foods; that becomes more fragile as the number of restaurants grows.

### 4. Keep dietary comparisons, but change the language and data model

Use `listed` or `flagged` language for allergen data until the pipeline can distinguish true, false, and unknown. Preserve nutrition snapshots or at least `observed_at`/source-version history before publishing historical nutrition trends. Treat zero nutrition from an absent source fact as unknown rather than a real zero.

## Further Questions

- What date should mark the first true in-session cohort? Store an academic-period tag instead of inferring it later from restaurant count.
- Should convenience retail locations with an all-day `Open` meal be compared with dining halls, or shown in a separate venue type?
- Does the source explicitly distinguish allergen absence from missing allergen data? If not, public copy must remain conservative.
- Should menu-history percentages use the last N observed menu days, the last N calendar days, or same-weekday comparisons? For users, observed menu days are the most honest default.

## Caveats and Assumptions

- Summer-break and camp schedules dominate the sample; weekday/weekend differences are not semester norms.
- July 1 is seed data and is excluded from live findings.
- July 18 and July 19 contain meal windows but no stored stations or foods.
- Failed scrape attempts preserve the last successful version of an already-imported service date, so attempt success rate is not the same as end-user date coverage.
- Re-importing a service date replaces its restaurant/meal/station hierarchy; intra-day version history is not retained.
- Nutrition facts and dietary/allergen flags are globally updated per food ID; historical menu rows join to the latest-known attributes.
- The report recommendations are product decisions informed by the current evidence, not causal claims about student behavior.